<h1>🧪 Biofilter — Report: <code>annotation_master_protein</code></h1>

Everything the bundle knows about a list of proteins: UniProt record, isoform resolution, Pfam domains by type, and what the protein is linked to.

### 1. Open a bundle

In [ ]:
from biofilter import Biofilter

# A bundle is a directory — the one holding manifest.json.
# Leave as None to use `[database] bundle` from .biofilter.toml.
BUNDLE = None
REPORT = "annotation_master_protein"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)
bf

### 2. What the report offers

In [ ]:
print("columns:")
for column in bf.report.available_columns(REPORT):
    print(" ", column)

print("\nexample input:")
print(bf.report.example_input(REPORT))

In [ ]:
print(bf.report.explain(REPORT))

### 3. Run it

Proteins resolve by accession, by entry name, or by an **isoform**
accession — section 4 is about what happens then.

In [ ]:
proteins = [
    "P04637",       # TP53, canonical accession
    "TP53_HUMAN",   # the same protein, by entry name
    "P04637-2",     # an isoform of it
    "NOT_A_PROT",   # kept, with status='not_found'
]

result = bf.report.run(REPORT, input_data=proteins)
df = result.to_pandas()
df[["input_value", "protein_id", "entity_id", "canonical_entity_id",
    "input_is_isoform", "status"]]

### 4. Two entity ids, and they can differ

A protein with isoforms has an entity per isoform as well as one for the
canonical sequence.

| column | what it is |
| --- | --- |
| `entity_id` | the entity the **input** matched — may be an isoform |
| `canonical_entity_id` | the entity the **annotation** describes |

An isoform entity carries almost nothing on its own, so the report follows
it to the canonical protein before annotating. Reporting the isoform's zero
relationships would be technically true and practically useless. The `note`
says so whenever the two ids differ.

In [ ]:
df[["input_value", "entity_id", "canonical_entity_id", "isoform_count", "note"]]

`isoform_count` counts isoforms, not entities: a protein with
`isoform_count = 3` has four entities.

### 5. Pfam domains, by type

In [ ]:
def as_list(value):
    """Nullable list column to a Python list. `value or []` raises on an array."""
    return [] if value is None else list(value)


for _, row in df[df["status"] == "ok"].iterrows():
    print(f"{row['protein_id']}  ({row['pfam_total_count']} domains)")
    for entry in as_list(row["pfam_ids_by_type"]):
        print(f"    {entry['type']:<10} {list(entry['ids'])[:6]}")
    print("   ", (row["function"] or "")[:90])
    print()

Counts are **distinct accessions**: a domain appearing twice in a
sequence is one accession. `pfam_total_count` is the sum across types.

### 6. Every protein in the bundle

In [ ]:
import time

started = time.perf_counter()
everything = bf.report.run(REPORT, input_data="__ALL__", include_pfam_summary=False)
catalog = everything.to_pandas()

print(f"{everything.num_rows:,} protein entities in {time.perf_counter() - started:.1f}s")
print("isoform inputs:", int(catalog["input_is_isoform"].fillna(False).sum()))
catalog["status"].value_counts()

### 7. Export

CSV by default, with a `.provenance.json` beside it naming the bundle the ids came from.

In [ ]:
for path in result.write("annotation_master_protein.csv"):
    print(path)

### 8. The same thing on the command line

```bash
biofilter report run --report-name annotation_master_protein \\
    --input ... \\
    --output out.csv
```

### 9. Quick QA

In [ ]:
expected = list(bf.report.available_columns(REPORT))
missing = [c for c in expected if c not in df.columns]

print("missing columns:", missing or "none")
print("unresolved inputs:", int((df["status"] == "not_found").sum()))
print("bundle:", result.provenance["bundle_id"])
display(df.dtypes.to_frame("dtype"))